In [14]:
import json
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import re
from tqdm import tqdm
from typing import Optional, List, Tuple


# check if gpu is available
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

Using device: GPU


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [15]:
MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

def load_model(model_name: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda" if torch.cuda.is_available() else None,
        torch_dtype="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # se manca pad token, usiamo eos come pad
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # impostiamo padding_side='left' per decoder-only mode
    tokenizer.padding_side = "left"

    # Creiamo la pipeline (la pipeline userà il tokenizer modificato)
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        dtype=torch.float16 ,#"auto"
        #torch_dtype=torch.float16 #"auto"
    )

    # Assicuriamoci che la pipeline conosca il pad_token_id (alcune versioni lo richiedono esplicitamente)
    pipe.tokenizer.pad_token_id = tokenizer.pad_token_id

    return pipe, model, tokenizer

def destroy_model(model, tokenizer):
    if model is not None:
        del model
    
    if tokenizer is not None:
        del tokenizer

    torch.cuda.empty_cache()

In [16]:
def clear_output(text: str) -> Optional[dict]:
    """
    Estrae e ripara il JSON tra <JSON> e </JSON> da un testo.
    """
    if not text:
        return None

    # Trova il primo blocco <JSON> ... </JSON>
    match = re.search(r'<JSON\s*>(.*?)</JSON\s*>', text, re.DOTALL | re.IGNORECASE)
    if not match:
        return None

    json_str = match.group(1).strip()

    # Escape ritorni a capo e virgolette
    def escape_json_string(s):
        s = s.replace('\\', '\\\\')   # escape backslash
        s = s.replace('"', '\\"')     # escape double quotes
        s = s.replace('\n', '\\n')    # escape newlines
        s = s.replace('\r', '\\r')
        return s

    # Regex per trovare tutte le stringhe dentro virgolette e escaparle
    def escape_all_strings(json_text):
        def replacer(match):
            inner = match.group(1)
            return f'"{escape_json_string(inner)}"'
        return re.sub(r'"(.*?)"', replacer, json_text, flags=re.DOTALL)

    json_str = escape_all_strings(json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Errore nel parsing JSON: {e}")
        return None

In [17]:
result = """<JSON>
            {
                "turns": [
                    {
                        "u1": I'm so drunk I can barely stand",
                        "u2": "Pisser. Amusing joke or incident. You're not the first to get hammered and stumble around"
                    },
                    {
                        "u1": "I'm not drunk, I'm just pissed off. I'm so angry I could scream",
                        "u2": "Pissweak or weak as piss. Very weak. You're not even close to being angry, you're just drunk"
                    },
                    {
                        "u1": "I'm not wasting time, I'm just pissing around. I'm taking a break from the stress",
                        "u2": "Pissing around. Wasting time. You're just avoiding the issue, you're not taking a break from anything"
                    }
                ],
                "last_turn": {
                    "u1": "Piss. Alcohol
Pissed. Drunk
Pissed off. Angry.
Pissing. Urinating
Pissing down. Heavy rain
Taking the piss. Having a joke at someone's expense
Taking a piss. Urinating",
                    "u2": "Pisser. Amusing joke or incident.
Pissing around. Wasting time.
Pissweak or weak as piss. Very weak
Piss off. Go away.
Some of these aren't unique to Aus but what a versatile word!"
                }
            }
            </JSON>

            Note: The first message from u1 is not explicitly stated in the prompt, but based on the context of the conversation, it is assumed to be a message that sets the tone for the conversation, which is that u1 is drunk and possibly joking around. The first message from u1 is therefore "I'm so drunk I can barely stand". The first message from u2 is a response to u1's message, which is a lighthearted and humorous take on the situation"""

clear_output(result)

Errore nel parsing JSON: Expecting value: line 4 column 31 (char 81)


In [ ]:
def generate_batch_output(pipe, tokenizer, turn: List[Tuple[str, str]], temperature: float = 0.1, batch_size: int = 1):
    """
    Genera output in batch per una lista di coppie di messaggi.
    
    Args:
        pipe: il modello di generazione (pipeline o callable)
        tokenizer: tokenizer associato al modello
        turn: lista di tuple (post, reply)
        temperature: parametro di sampling
        batch_size: dimensione del batch
    
    Returns:
        Lista di tuple (prompt, model_output, dialogue_dict)
    """
    prompts = []
    for post, reply in turn:
        prompt = f"""
            Given this pair of messages extracted from social media:

            u1: {post}
            u2: {reply}

            Generate the 3 previous dialogue turns (each turn consisting of a message from u1 and a message from u2) that logically lead up to the given messages. Make sure to:
            - Maintain the original writing style of both u1 e u2.
            - Preserve the sentiment and tone of the conversation.
            - Keep the dialogue coherent and consistent with the context of the given messages.

            Return the result in the following JSON format, including both the previous 3 turns and the given messages as the last turn, enclosed in <JSON> tags:

            <JSON>
            {{
                "turns": [
                    {{
                        "u1": "First message from user1 in the earliest turn",
                        "u2": "First message from user2 in response"
                    }},
                    {{
                        "u1": "Second message from user1",
                        "u2": "Second message from user2"
                    }},
                    {{
                        "u1": "Third message from user1",
                        "u2": "Third message from user2"
                    }},
                    {{
                        "u1": "{post}",
                        "u2": "{reply}"
                    }}
                ]
            }}
            </JSON>

            Response:
            <JSON>
            {{
                "turns": [
                    {{
                        "u1": "
            """
        prompts.append(prompt)

    # Genera i risultati dal modello
    outputs = pipe(
        prompts,
        max_new_tokens=1024,
        do_sample=temperature > 0,
        temperature=temperature,
        return_full_text=False,
        batch_size=batch_size,
        pad_token_id=tokenizer.eos_token_id
    )


    results = []
    for prompt, out in zip(prompts, outputs):
        entry = out[0]
        model_output = entry['generated_text'].strip()
        #print("Model output:", model_output)
        initial_output = """<JSON>{"turns": [{"u1":\""""
        dialogue = clear_output(f"{initial_output}{model_output}")
        results.append((prompt, model_output, dialogue))

    return results


In [19]:
def generate_dialogue(dataset_df: pd.DataFrame):
    for model_name in MODELS:
        print(f"Processing model: {model_name}")
        pipe, model, tokenizer = load_model(model_name)
        print("Model and tokenizer loaded.")

        output_file = f'test1_{model_name.replace("/", "_")}.json'
        output_rows = []

        # --- carica dati già esistenti, se presenti ---
        if os.path.exists(output_file):
            try:
                with open(output_file, 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                existing_ids = set([row['unique_id'] for row in existing_data])
                print(f"Found {len(existing_ids)} already processed entries in {output_file}")
                # aggiungi i dati esistenti a output_rows
                output_rows.extend(existing_data)
            except Exception as e:
                print(f"Error loading existing JSON: {e}")
                existing_ids = set()
        else:
            existing_ids = set()

        # filtra dataset_df per saltare già processati
        dataset_df = dataset_df[~dataset_df['unique_id'].isin(existing_ids)]
        print(f"{len(dataset_df)} new entries to process")

        batch_size = 1
        for start in tqdm(range(0, len(dataset_df), batch_size), desc=f"Running {model_name}"):
            batch_df = dataset_df.iloc[start:start+batch_size]

            post = batch_df['post'].tolist()
            post = [re.sub(r'@USER', '', p) for p in post]

            reply = batch_df['reply'].tolist()
            reply = [re.sub(r'@USER', '', r) for r in reply]

            turn = list(zip(post, reply))

            results = generate_batch_output(pipe, tokenizer, turn)

            cleaned_results = []
            for i, (prompt, model_output, dialogue) in enumerate(results):
                invalid_inferences = 0
                while dialogue is None and invalid_inferences < 3:
                    invalid_inferences += 1
                    prompt, model_output, dialogue = generate_batch_output(pipe, tokenizer, [turn[i]])[0]

                if invalid_inferences >= 3:
                    print(f"⚠️  Warning: Max retries reached for entry {batch_df.iloc[i]['unique_id']}. Saving last output anyway.")
                    dialogue = {"error": "Max retries reached"}
                    model_output = {"error": "Max retries reached"}

                cleaned_results.append((prompt, model_output, dialogue, invalid_inferences))

            for row, (prompt, model_output, dialogue, invalid_inferences) in zip(batch_df.itertuples(), cleaned_results):
                enriched = row._asdict()
                enriched['prompt'] = prompt
                enriched['model_output'] = model_output
                enriched['dialogue'] = dialogue
                enriched['invalid_inferences'] = invalid_inferences
                output_rows.append(enriched)

            # salva ad ogni batch
            pd.DataFrame(output_rows).to_json(
                output_file,
                orient='records',
                indent=2,
                force_ascii=False
            )
            print(f"✔ Batch {start//batch_size + 1} salvato → {output_file}")

        destroy_model(model, tokenizer)
        print("Model and tokenizer destroyed.")

In [20]:
# check if exists and open unified corpus in 1_datasets_merge\output\unified_corpus.jsonù
file_path = '../1_datasets_merge/output/unified_corpus.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        unified_corpus = json.load(f)
    print(f"Loaded unified corpus with {len(unified_corpus)} entries")
    df = pd.DataFrame(unified_corpus)
except Exception as e:
    print(f"Error loading unified corpus: {e}")
    raise e

# get only row with language == 'en' and dataset == 'MultiPICo'
df_en_multipico = df[(df['language'] == 'en') & (df['dataset_name'] == 'MultiPICo')]
# show number of entries
print(f"Filtered MultiPICo English entries: {len(df_en_multipico)}")

# get only 700 entries for testing
df_en_multipico = df_en_multipico.head(100)
print(f"Using {len(df_en_multipico)} entries for testing")


generate_dialogue(df_en_multipico)

Loaded unified corpus with 25814 entries
Filtered MultiPICo English entries: 2999
Using 100 entries for testing
Processing model: meta-llama/Llama-3.2-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.65s/it]
Device set to use cuda


Model and tokenizer loaded.
Found 10 already processed entries in test1_meta-llama_Llama-3.2-3B-Instruct.json
90 new entries to process


Running meta-llama/Llama-3.2-3B-Instruct:   1%|          | 1/90 [06:12<9:12:37, 372.56s/it]

Model output: , Appudu em cheyaali?",
                        "u2": "Go girl anali"
                    },
                    {
                        "u1": " Same tweet reverse lo oka ammayi veste, Appudu em cheyaali?",
                        "u2": "   Go girl anali"
                    },
                    {
                        "u1": " Same tweet reverse lo oka ammayi veste, Appudu em cheyaali?",
                        "u2": "   Go girl anali"
                    },
                    {
                        "u1": " Same tweet reverse lo oka ammayi veste, Appudu em cheyaali?",
                        "u2": "   Go girl anali"
                    }
                ]
            }
            </JSON>

            Here is the code to generate the dialogue turns:

            ```python
            def generate_dialogue(turns):
                #... (rest of the code remains the same)
                return turns

            def generate_previous_turns():
                #... 

...

Model output: I just saw the most beautiful sunset today! It was so peaceful and calming. I wish I could bottle up that feeling and share it with everyone.",
                        "u2": "That sounds amazing! I'm so glad you got to experience that. It's moments like those that make life truly special."
                    },
                    {
                        "u1": "I know, right?! It's like the world just stops for a second and everything feels right with the universe. I'm so grateful for those moments.",
                        "u2": "I completely agree! Those moments are what make life worth living. They give us the strength to keep going, even when things get tough."
                    },
                    {
                        "u1": "Exactly! And it's not just about the big moments, it's about the little things too. Like a good cup of coffee or a beautiful sunset. Those are the things that make life beautiful.",
                        "u2": "I love that! It's t

Running meta-llama/Llama-3.2-3B-Instruct:  30%|███       | 27/90 [2:39:54<3:40:37, 210.12s/it]

Model output: I'm so sick of all the hate speech going around here. Can't we all just get along?",
                        "u2": "I agree, but sometimes it feels like we're just preaching to the choir."
                    },
                    {
                        "u1": "Exactly! And it's not just the hate speech, it's the ignorance and close-mindedness too.",
                        "u2": "I know, it's like some people are just not willing to listen or learn."
                    },
                    {
                        "u1": "Well, I think we need to call out the hypocrisy and double standards when we see them.",
                        "u2": "But don't you think that can come across as accusatory and alienate people?"
                    },
                    {
                        "u1": "Muslims are the few sane ppl left in this country",
                        "u2": "Oh please, let’s not get carried away with such moronic rhetoric!"
                    }
      

Running meta-llama/Llama-3.2-3B-Instruct:  30%|███       | 27/90 [2:47:13<6:30:12, 371.63s/it]


KeyboardInterrupt: 